# 🌍 MULTI-DATASET - ALL 7 Bi-encoders on 9 DatasetsTest ALL models on: AG News, DBpedia-14, Yahoo Answers, Banking77, Twitter Financial, 20 Newsgroups, GoEmotions, IMDB, SST-2**Note**: This notebook uses the actual `src/encoders.py` file from the project. All 7 models are already supported.

## Setup: Option 1 - Clone from GitHub (RECOMMENDED)GitHub'dan clone edince tüm güncellemeler otomatik gelir!

In [ ]:
# 1. Mount Drive (sonuçlar Drive'a kaydedilecek)from google.colab import drivedrive.mount('/content/drive')# 2. GitHub'dan clone et (en güncel kod!)!git clone https://github.com/EngindalgaMaku/zeroshot-text-classification-benchmark-.git%cd zeroshot-text-classification-benchmark-# 3. Sonuçları Drive'a kaydetmek için symlink oluşturimport osdrive_results = '/content/drive/MyDrive/zeroshot_results'os.makedirs(drive_results, exist_ok=True)# results klasörünü Drive'a yönlendirif os.path.exists('results'):    !rm -rf results!ln -s {drive_results} resultsprint(f"✅ Results will be saved to: {drive_results}")# 4. Paketleri yükle!pip install -q -r requirements.txtprint("✅ Setup complete!")

## Setup: Option 2 - Use Drive (Manuel Upload)Sadece Drive'a yüklediyseniz kullanın

In [ ]:
from google.colab import drivedrive.mount('/content/drive')

In [ ]:
import osos.chdir('/content/drive/MyDrive/zero_shot_colab_final')print("✅", os.getcwd())

In [ ]:
!pip install -q datasets sentence-transformers scikit-learn pandas pyyaml matplotlib seaborn transformers accelerate

In [ ]:
import torchprint(f"GPU: {torch.cuda.is_available()}")if torch.cuda.is_available():    print(f"{torch.cuda.get_device_name(0)} - {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 📊 9 Datasets × 7 Models = 63 Experiments!### Datasets:1. **AG News**: 4 classes (World, Sports, Business, Tech) - News classification2. **DBpedia-14**: 14 classes (Company, Athlete, Plant, etc.) - Entity classification3. **Yahoo Answers**: 10 classes (Science, Health, Sports, etc.) - Question classification  4. **Banking77**: 77 classes (card issues, transfers, etc.) - Intent classification (VERY HARD!)5. **Twitter Financial**: 3 classes (bearish, bullish, neutral) - Financial sentiment6. **20 Newsgroups**: 20 classes (alt.atheism, comp.graphics, etc.) - Classic benchmark7. **GoEmotions**: 28 classes (joy, anger, sadness, etc.) - Fine-grained emotion classification8. **IMDB**: 2 classes (negative, positive) - Movie review sentiment (LONG TEXTS) 🆕9. **SST-2**: 2 classes (negative, positive) - Movie review sentiment (SHORT TEXTS, GLUE benchmark) 🆕### ALL 7 Models:1. MPNet (420M)2. BGE-M3 (567M)3. E5-large (560M, multilingual)4. Qwen3 (8B)5. Jina v5 nano (33M)6. INSTRUCTOR-large (335M)7. Nomic Embed v2 MoE (300M, MoE architecture) 🆕

In [ ]:
import yamlfrom pathlib import Path# 9 English datasetsdatasets = [    ("ag_news", "text", "label", "test", 1000),    ("dbpedia_14", "content", "label", "test", 1000),    ("yahoo_answers_topics", "best_answer", "topic", "test", 1000),    ("banking77", "text", "label", "test", 1000),    ("zeroshot/twitter-financial-news-sentiment", "text", "label", "validation", 1000),    ("SetFit/20_newsgroups", "text", "label", "test", 2000),    ("go_emotions", "text", "labels", "test", 1000),    ("imdb", "text", "label", "test", 1000),    ("sst2", "sentence", "label", "validation", 1000),]models = [    ("sentence-transformers/all-mpnet-base-v2", "mpnet"),    ("BAAI/bge-m3", "bge"),    ("intfloat/multilingual-e5-large", "e5"),    ("Qwen/Qwen3-Embedding-8B", "qwen3"),    ("jinaai/jina-embeddings-v5-text-nano", "jina_v5"),    ("hkunlp/instructor-large", "instructor"),    ("nomic-ai/nomic-embed-text-v2-moe", "nomic"),]Path("experiments/multi_dataset").mkdir(parents=True, exist_ok=True)for ds_name, text_col, label_col, split, max_samples in datasets:    for model_name, model_short in models:        ds_clean = ds_name.replace("/", "_").replace("-", "_")        exp_name = f"{ds_clean}_{model_short}"                             config = {            "experiment_name": exp_name,            "dataset": {                "name": ds_name,                "split": split,                "text_column": text_col,                "label_column": label_col,                "max_samples": max_samples            },            "task": {                "type": "zero_shot_classification",                "label_mode": "description",                "language": "en"            },            "models": {                "biencoder": {"provider": "hf", "name": model_name},                "reranker": None            },            "pipeline": {"mode": "biencoder", "normalize_embeddings": True, "batch_size": 16 },            "evaluation": {"metrics": ["accuracy", "macro_f1", "per_class_f1"]},            "output": {"save_predictions": True, "save_metrics": True, "output_dir": "results/raw"}        }        with open(f"experiments/multi_dataset/{exp_name}.yaml", "w") as f:            yaml.dump(config, f)        print(f"✅ {exp_name}")print(f"\n📊 Total: {len(datasets) * len(models)} experiments created! (9 datasets × 7 models = 63)")

In [ ]:
# ⚙️ SETTINGSSKIP_EXISTING = True  # Set to False to re-run all experiments# Run all experimentsimport globconfigs = sorted(glob.glob("experiments/multi_dataset/*.yaml"))print(f"Found {len(configs)} experiments")if SKIP_EXISTING:    print("Mode: --skip-existing (will skip if results exist)")    print("💡 To re-run all experiments, set SKIP_EXISTING = False\n")else:    print("Mode: OVERWRITE (will re-run all experiments)\n")results = []for i, config in enumerate(configs, 1):    exp_name = config.split("\\")[-1].replace(".yaml", "")    print(f"\n{'='*70}")    print(f"[{i}/{len(configs)}] {exp_name}")    print(f"{'='*70}")    try:        cmd = f"python main.py --config {config}"        if SKIP_EXISTING:            cmd += " --skip-existing"        !{cmd}        results.append((exp_name, "✅"))    except Exception as e:        print(f"ERROR: {e}")        results.append((exp_name, "❌"))print("\n" + "="*70)print("SUMMARY")print("="*70)success = sum(1 for _, s in results if s == "✅")print(f"Success: {success}/{len(results)}\n")for exp, status in results:    print(f"{status} {exp}")

In [ ]:
# Load and organize resultsimport jsonimport pandas as pdfrom pathlib import Pathfiles = list(Path("results/raw").glob("*_metrics.json"))data = []for f in files:    if any(ds in f.name for ds in ["ag_news", "dbpedia", "yahoo", "banking", "twitter", "financial", "20_newsgroups", "SetFit", "go_emotions", "goemotions", "IMDB", "SST-2"]):        with open(f) as fp:            m = json.load(fp)                exp_name = m.get("experiment_name", f.stem)        if "ag_news" in exp_name:            dataset = "AG News"        elif "dbpedia" in exp_name:            dataset = "DBpedia-14"        elif "yahoo" in exp_name:            dataset = "Yahoo Answers"        elif "banking" in exp_name:            dataset = "Banking77"        elif "twitter" in exp_name or "financial" in exp_name:            dataset = "Twitter Financial"        elif "20_newsgroups" in exp_name or "SetFit" in exp_name:            dataset = "20 Newsgroups"        elif "go_emotions" in exp_name or "goemotions" in exp_name:            dataset = "GoEmotions"        elif "imdb" in exp_name:            dataset = "IMDB"        elif "sst2" in exp_name:            dataset = "SST-2"             else:            continue                if "mpnet" in exp_name:            model = "MPNet"        elif "jina_v5" in exp_name:            model = "Jina v5"        elif "qwen3" in exp_name:            model = "Qwen3"        elif "bge" in exp_name:            model = "BGE-M3"        elif "e5" in exp_name:            model = "E5-large"        elif "instructor" in exp_name:            model = "INSTRUCTOR"        elif "nomic" in exp_name or "arctic" in exp_name:            model = "Nomic-MoE"        else:            model = "Unknown"                data.append({            "dataset": dataset,            "model": model,            "samples": m["num_samples"],            "num_classes": len(set([v for v in m["classification_report"].keys() if v.isdigit()])),            "accuracy": m["accuracy"] * 100,            "macro_f1": m["macro_f1"] * 100,        })df = pd.DataFrame(data)print("\n" + "="*70)print("ALL RESULTS (7 MODELS × 9 datasets)")print("="*70)print(df.to_string(index=False))df.to_csv("results/MULTI_DATASET_RESULTS.csv", index=False)print("\n✅ results/MULTI_DATASET_RESULTS.csv")

In [ ]:
# Pivot tablespivot_acc = df.pivot(index="model", columns="dataset", values="accuracy")pivot_f1 = df.pivot(index="model", columns="dataset", values="macro_f1")print("\n" + "="*70)print("ACCURACY (%)")print("="*70)print(pivot_acc.to_string())print("\n" + "="*70)print("MACRO F1 (%)")print("="*70)print(pivot_f1.to_string())

In [ ]:
# Visualizeimport matplotlib.pyplot as pltimport seaborn as snsfig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 7))# F1 scores by datasetpivot_f1.plot(kind='bar', ax=ax1)ax1.set_ylabel("Macro F1 (%)")ax1.set_title("Model Performance Across 9 datasets (7 Models)")ax1.legend(title="Dataset", bbox_to_anchor=(1.05, 1), loc='upper left')ax1.set_ylim(0, 100)ax1.grid(axis='y', alpha=0.3)# Heatmapsns.heatmap(pivot_f1, annot=True, fmt=".1f", cmap="YlGnBu", ax=ax2, cbar_kws={'label': 'Macro F1 (%)'})ax2.set_title("F1 Score Heatmap (9 datasets)")ax2.set_xlabel("Dataset")ax2.set_ylabel("Model")plt.tight_layout()plt.savefig("results/plots/multi_dataset_comparison.png", dpi=300, bbox_inches="tight")plt.show()print("✅ results/plots/multi_dataset_comparison.png")

### Difficulty Ranking:1. **IMDB** (2 classes, long texts) - EASY-MEDIUM 🆕2. **SST-2** (2 classes, short texts) - EASY 🆕3. **Twitter Financial** (3 classes) - EASY4. **AG News** (4 classes) - EASY5. **Yahoo Answers** (10 classes) - MEDIUM6. **DBpedia** (14 classes) - MEDIUM-HARD7. **20 Newsgroups** (20 classes) - HARD8. **GoEmotions** (28 classes) - HARD (fine-grained emotions)9. **Banking77** (77 classes) - VERY HARD!### Questions to Answer:- Which model is most consistent across datasets?- Does model size matter more on harder tasks?- How does performance degrade with more classes?- Best all-rounder model?- **NEW**: How do models perform on binary sentiment (IMDB vs SST-2)?- **NEW**: Does text length affect sentiment classification performance?

In [ ]:
# Calculate average performance across all datasetsavg_perf = df.groupby("model")[["accuracy", "macro_f1"]].mean().sort_values("macro_f1", ascending=False)print("\n" + "="*70)print("AVERAGE PERFORMANCE ACROSS ALL 9 datasets")print("="*70)print(avg_perf.to_string())print("\n🏆 Best all-rounder:", avg_perf.index[0])

In [ ]:
# Performance by dataset difficulty (number of classes)fig, ax = plt.subplots(figsize=(12, 6))for model in df["model"].unique():    model_data = df[df["model"] == model].sort_values("num_classes")    ax.plot(model_data["num_classes"], model_data["macro_f1"], marker='o', label=model, linewidth=2)ax.set_xlabel("Number of Classes")ax.set_ylabel("Macro F1 (%)")ax.set_title("Model Performance vs Dataset Difficulty (9 datasets)")ax.legend()ax.grid(alpha=0.3)plt.tight_layout()plt.savefig("results/plots/performance_vs_difficulty.png", dpi=300, bbox_inches="tight")plt.show()print("✅ results/plots/performance_vs_difficulty.png")

In [ ]:
# NEW: Sentiment Analysis Comparison (IMDB vs SST-2 vs Twitter Financial)sentiment_datasets = ["IMDB", "SST-2", "Twitter Financial"]sentiment_df = df[df["dataset"].isin(sentiment_datasets)]if not sentiment_df.empty:    fig, ax = plt.subplots(figsize=(12, 6))        sentiment_pivot = sentiment_df.pivot(index="model", columns="dataset", values="macro_f1")    sentiment_pivot.plot(kind='bar', ax=ax)        ax.set_ylabel("Macro F1 (%)")    ax.set_title("Sentiment Classification Performance Comparison")    ax.set_xlabel("Model")    ax.legend(title="Dataset")    ax.grid(axis='y', alpha=0.3)    ax.set_ylim(0, 100)        plt.tight_layout()    plt.savefig("results/plots/sentiment_comparison.png", dpi=300, bbox_inches="tight")    plt.show()    print("✅ results/plots/sentiment_comparison.png")        print("\n" + "="*70)    print("SENTIMENT CLASSIFICATION ANALYSIS")    print("="*70)    print(sentiment_pivot.to_string())    print("\nKey Insights:")    print("- IMDB: Long-form movie reviews (avg 233 tokens)")    print("- SST-2: Short-form movie reviews (avg 19 tokens, GLUE benchmark)")    print("- Twitter Financial: Financial news sentiment (avg 20 tokens)")

## ✅ DONE!### Key Findings:- **63 experiments** completed (7 models × 9 datasets)- **Diverse English domains**: News, Entities, Q&A, Banking, Financial Sentiment, Forums, Emotions- **Robustness test** across diverse text types and domains- **Scalability** from 3 to 77 classes (3, 4, 10, 14, 20, 77)- **Best all-rounder** identified- **Model size range**: 33M (Jina v5) to 8B (Qwen3)### For Paper:- Model consistency analysis- Size vs performance trade-off- Domain transfer ability- Recommendations by use case- Classic benchmark (20 Newsgroups) performance